# 2장. 이야기를 숫자로: 데이터와 글자 사전

모델이 읽을 이야기를 불러오고, 글자를 번호로 바꾸는 사전을 만들어요. 이 장은 **GPU가 필요 없어요.** (런타임 유형: CPU)

셀은 **위에서부터 차례로** 실행하세요. 뒤의 셀이 앞의 셀에서 만든 값을 이어서 씁니다.

**이 장이 끝나면**: 드라이브에 `vocab.json`(글자 사전)이 저장됩니다.


### 준비
이 책의 실습 패키지를 설치합니다. 2.4절에서 한글 2,350자를 꺼낼 때 씁니다.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm


## 2.1 모델이 읽을 이야기: Tiny-Ko-Stories

Tiny-Ko-Stories는 어린이 이야기 약 200만 편으로 된 한국어 데이터셋입니다(psymon, CC BY 4.0).
전체를 내려받지 않고 앞에서 3,000편만 읽어 `stories` 리스트에 모읍니다.

**핵심 코드**: `stories.append(story)`


In [ ]:
from datasets import load_dataset

dataset = load_dataset("psymon/Tiny-Ko-Stories", split="train", streaming=True)

stories = []
for row in dataset.take(3000):
    story = row["text"].strip()
    stories.append(story)

print("이야기 수:", len(stories))
print(stories[0])


이야기 하나의 길이는 평균 몇 글자일까요? 글자 수를 하나씩 더해 봅니다.


In [ ]:
total = 0
for story in stories:
    total = total + len(story)

print("전체 글자 수:", total)
print("평균 글자 수:", total // len(stories))


## 2.2 이야기의 끝을 알려 주는 표시

이야기들을 이어 붙일 때 사이에 `§`를 넣습니다. 모델은 이 표시를 보며 **이야기가 어디서 끝나는지**를 배우고,
나중에 챗봇은 `§`가 나오면 말을 멈춥니다. 이야기 안에도 문단 줄바꿈이 있어서 줄바꿈으로는 끝을 알 수 없습니다.

**핵심 코드**: `text = END.join(stories) + END`


In [ ]:
END = "§"
text = END.join(stories) + END

print("이어 붙인 글자 수:", len(text))
print("끝 표시 개수:", text.count(END))
print(text[:300])


## 2.3 글자를 번호로 바꾸는 사전

모델은 글자를 직접 계산하지 못하고 숫자만 다룹니다. 그래서 글자마다 번호를 붙인 사전이 필요합니다.
가장 쉬운 방법은 이야기에 나온 글자를 모두 모아 번호를 붙이는 것입니다.

**핵심 코드**: `chars = sorted(set(text))`


In [ ]:
chars = sorted(set(text))
print("이야기에 나온 글자 수:", len(chars))
print(chars[:60])


그런데 이 방법에는 문제가 있습니다. **데이터가 바뀌면 사전도 바뀝니다.** 이야기를 더 읽혀 새 글자가 나오면 뒤 글자들의 번호가 밀립니다. 다음 절에서 이 문제를 풉니다.


## 2.4 언제나 같은 사전: 고정 글자 사전

자주 쓰는 한글 **2,350자**에 영문·숫자·기호를 더해 **데이터와 상관없이 늘 같은 사전**을 만듭니다.
한글 2,350자는 패키지의 `hangul_2350()`으로 꺼냅니다.

**핵심 코드**: `vocab_chars = ["\n"] + ascii_chars + extra + hangul + [END]`


In [ ]:
from modu_llm import hangul_2350

hangul = hangul_2350()
print("한글:", len(hangul))
print(hangul[:10], "...", hangul[-5:])


In [ ]:
ascii_chars = []
for number in range(32, 127):
    ascii_chars.append(chr(number))

extra = list("…‘’“”·")

vocab_chars = ["\n"] + ascii_chars + extra + hangul + [END]
print("사전 전체:", len(vocab_chars))


이 사전이 이야기 3,000편의 글자를 얼마나 담고 있는지 세어 봅니다.


In [ ]:
vocab_set = set(vocab_chars)

count = 0
for ch in text:
    if ch in vocab_set:
        count = count + 1

print("사전에 있는 글자:", count, "/", len(text))
print("사전에 없는 글자:", sorted(set(text) - vocab_set))


## 2.5 글자와 번호를 오가는 encode와 decode

`stoi`는 글자 → 번호, `itos`는 번호 → 글자 사전입니다. `encode`는 글을 번호 목록으로, `decode`는 번호 목록을 글로 바꿉니다.
사전에 없는 글자는 **건너뜁니다**.

**핵심 코드**: `if ch in stoi:`


In [ ]:
stoi = {}
itos = {}
for i, ch in enumerate(vocab_chars):
    stoi[ch] = i
    itos[i] = ch

print(stoi["가"], stoi[" "], stoi[END])
print(itos[102])


In [ ]:
def encode(s):
    ids = []
    for ch in s:
        if ch in stoi:
            ids.append(stoi[ch])
    return ids

def decode(ids):
    s = ""
    for i in ids:
        s = s + itos[i]
    return s

print(encode("어느 날"))
print(decode(encode("어느 날")))
print(decode(encode("똠양꿍 먹고 싶어요")))


## 2.6 사전을 파일로 저장하기

구글 드라이브를 연결하고 사전을 `vocab.json`으로 저장합니다. 드라이브 연결 창이 뜨면 허용하세요.

**핵심 코드**: `json.dump(vocab_chars, f, ensure_ascii=False)`


In [ ]:
import json
from modu_llm import mount_drive

folder = mount_drive()
path = folder + "/vocab.json"

with open(path, "w", encoding="utf-8") as f:
    json.dump(vocab_chars, f, ensure_ascii=False)

print("저장:", path)


이 사전은 고정되어 있으므로, 실습 패키지의 `Vocab.build()`도 **똑같은 사전**을 만듭니다. 3장부터는 이 한 줄로 사전을 준비합니다. 정말 같은지 확인해 봅시다.


In [ ]:
from modu_llm import Vocab

vocab = Vocab.build()
print("패키지 사전과 같은가요?", vocab.chars == vocab_chars)
print(vocab.encode("어느 날"))


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

- 내 이름의 글자가 사전에 모두 있는지 확인해 보세요.
- 이야기에 가장 많이 나온 글자 20개를 보세요. 어떤 글자가 많을까요?


In [ ]:
name = "홍길동"   # 내 이름으로 바꿔 보세요
for ch in name:
    print(ch, ch in stoi)

print(encode(name))
print(decode(encode(name)))


In [ ]:
from collections import Counter

print(Counter(text).most_common(20))


## 실제 LLM에서는

- 실제 LLM은 글자 대신 자주 함께 나오는 글자 묶음(하위 단어)을 토큰으로 씁니다. 사전 크기는 수만~수십만 개이고, 데이터에서 자동으로 만듭니다(BPE 등).
- 하위 단어를 쓰면 같은 글을 더 적은 토큰으로 나타낼 수 있어, 같은 문맥 길이에 더 많은 내용을 담습니다.
- 실제 LLM의 사전도 한 번 정하면 바꾸지 않습니다.
